# PAS Validation Harness
## 70/30 Split Validation & Lift Analysis

Tests whether composite_score predicts:
1. Loss cost (kpi_total_loss_cost)
2. Severity (kpi_total_severity)

Metrics:
- Average loss cost by score band
- Gini coefficient
- Decile lift (top decile vs bottom)

## Cell 1: Load Data & Create 70/30 Split

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

csv_path = r'C:\Box\Box\BOX Subhashree Singh\Business\PAS\Input\physician_scores_20260429_141959.csv'

print("Loading data...")
df = pd.read_csv(csv_path, low_memory=False)
print(f"✅ Loaded {len(df):,} physicians\n")

# Random 70/30 split (stratified optional - can add if needed)
np.random.seed(42)  # For reproducibility
train_mask = np.random.rand(len(df)) < 0.70

train_df = df[train_mask].copy()
test_df = df[~train_mask].copy()

print(f"Train set: {len(train_df):,} physicians ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test set:  {len(test_df):,} physicians ({len(test_df)/len(df)*100:.1f}%)")
print(f"\nComposite score distribution (Train):")
print(train_df['composite_score'].value_counts().sort_index())

Loading data...
✅ Loaded 165,623 physicians

Train set: 115,808 physicians (69.9%)
Test set:  49,815 physicians (30.1%)

Composite score distribution (Train):
composite_score
2     1264
3    19378
4    38578
5    37527
6    16893
7     2166
8        2
Name: count, dtype: int64


## Cell 2: Average Loss Cost by Score Band (Test Set)

In [2]:
print("="*100)
print("LOSS COST ANALYSIS BY COMPOSITE SCORE BAND")
print("="*100)
print()

# Group test set by composite score band
loss_by_band = test_df.groupby('composite_score').agg({
    'kpi_total_loss_cost': ['count', 'mean', 'median', 'std'],
    'NPI': 'count'  # just for count
}).round(2)

loss_by_band.columns = ['Physicians', 'Avg Loss Cost', 'Median Loss Cost', 'Std Dev', '_']
loss_by_band = loss_by_band.drop('_', axis=1)

print("Test Set Performance by Band:")
print(loss_by_band)

# Calculate lift from band 1 (best) to band 10 (worst)
band_1_loss = test_df[test_df['composite_score'] == 2]['kpi_total_loss_cost'].mean()
band_high_loss = test_df[test_df['composite_score'] >= 7]['kpi_total_loss_cost'].mean()

if pd.notna(band_1_loss) and band_1_loss > 0:
    overall_lift = (band_high_loss / band_1_loss) if pd.notna(band_high_loss) else 0
    print(f"\nLift Summary:")
    print(f"  Low Risk (Band 2) Avg Loss Cost:  ${band_1_loss:,.2f}")
    print(f"  High Risk (Bands 7+) Avg Loss Cost: ${band_high_loss:,.2f}")
    print(f"  Lift (High/Low):                   {overall_lift:.2f}x")
else:
    print(f"\n⚠️  Insufficient data for lift calculation")

LOSS COST ANALYSIS BY COMPOSITE SCORE BAND

Test Set Performance by Band:
                 Physicians  Avg Loss Cost  Median Loss Cost    Std Dev
composite_score                                                        
2                       555           0.00               0.0       0.00
3                      8334          67.49               0.0    3927.02
4                     16483        2486.60               0.0   61029.32
5                     16266        5265.91               0.0   79263.02
6                      7257       18204.35               0.0  164383.04
7                       919       26405.11               0.0  115881.33
8                         1           0.00               0.0        NaN

⚠️  Insufficient data for lift calculation


## Cell 3: Decile Lift Analysis (Top vs Bottom)

In [3]:
print("\n" + "="*100)
print("DECILE LIFT ANALYSIS")
print("="*100)
print()

# Create deciles based on composite_score
test_df['decile'] = pd.qcut(test_df['composite_score'], q=10, labels=False, duplicates='drop')

decile_analysis = test_df.groupby('decile').agg({
    'composite_score': ['min', 'max'],
    'kpi_total_loss_cost': ['count', 'mean', 'sum'],
    'NPI': 'count'
}).round(2)

decile_analysis.columns = ['Score Min', 'Score Max', 'Physicians', 'Avg Loss Cost', 'Total Loss', '_']
decile_analysis = decile_analysis.drop('_', axis=1)
decile_analysis.index = ['Decile ' + str(i+1) for i in range(len(decile_analysis))]

print(decile_analysis)

# Calculate decile lift
if len(decile_analysis) >= 2:
    top_decile_loss = decile_analysis['Avg Loss Cost'].iloc[-1]  # Last decile (worst)
    bottom_decile_loss = decile_analysis['Avg Loss Cost'].iloc[0]  # First decile (best)
    
    if pd.notna(bottom_decile_loss) and bottom_decile_loss > 0:
        decile_lift = top_decile_loss / bottom_decile_loss
        print(f"\nDecile Lift:")
        print(f"  Top Decile (Worst Risk) Avg Loss Cost:    ${top_decile_loss:,.2f}")
        print(f"  Bottom Decile (Best Risk) Avg Loss Cost:  ${bottom_decile_loss:,.2f}")
        print(f"  Decile Lift (Top/Bottom):                 {decile_lift:.2f}x")
    else:
        print(f"\n⚠️  Bottom decile has insufficient loss data")


DECILE LIFT ANALYSIS

          Score Min  Score Max  Physicians  Avg Loss Cost    Total Loss
Decile 1          2          3        8889          63.28  5.624570e+05
Decile 2          4          4       16483        2486.60  4.098662e+07
Decile 3          5          5       16266        5265.91  8.565523e+07
Decile 4          6          6        7257       18204.35  1.321090e+08
Decile 5          7          8         920       26376.41  2.426629e+07

Decile Lift:
  Top Decile (Worst Risk) Avg Loss Cost:    $26,376.41
  Bottom Decile (Best Risk) Avg Loss Cost:  $63.28
  Decile Lift (Top/Bottom):                 416.82x


## Cell 4: Gini Coefficient

In [4]:
def calculate_gini(y_true, y_pred):
    """
    Calculate Gini coefficient.
    Gini = 2 * AUC - 1
    
    Interpretation:
    - 0.0 = No discriminative power (random)
    - 0.2-0.3 = Weak
    - 0.3-0.5 = Moderate
    - 0.5+ = Strong
    """
    # Remove NaN values
    valid_idx = ~(y_true.isna() | y_pred.isna())
    y_true_clean = y_true[valid_idx]
    y_pred_clean = y_pred[valid_idx]
    
    if len(y_true_clean) < 2:
        return np.nan
    
    # For Gini, we need binary outcome. Create binary: has_loss (>0) vs no_loss (=0)
    y_binary = (y_true_clean > 0).astype(int)
    
    # If all same value, can't calculate AUC
    if len(np.unique(y_binary)) < 2:
        return 0.0
    
    try:
        auc = roc_auc_score(y_binary, y_pred_clean)
        gini = 2 * auc - 1
        return gini
    except:
        return np.nan

print("\n" + "="*100)
print("GINI COEFFICIENT (Model Discrimination Power)")
print("="*100)
print()

# Calculate Gini for loss cost prediction
gini_loss = calculate_gini(test_df['kpi_total_loss_cost'], test_df['composite_score'])

print(f"Gini Coefficient (Loss Cost):")
print(f"  Value: {gini_loss:.4f}")
print()
print(f"Interpretation:")
if gini_loss < 0.2:
    print(f"  🔴 Very Weak - Model has minimal discriminative power")
elif gini_loss < 0.3:
    print(f"  🟡 Weak - Model has some discriminative power")
elif gini_loss < 0.5:
    print(f"  🟢 Moderate - Model discriminates reasonably well")
else:
    print(f"  🟢🟢 Strong - Model discriminates very well")

print(f"\n  (0.0 = no power, 1.0 = perfect discrimination)")


GINI COEFFICIENT (Model Discrimination Power)

Gini Coefficient (Loss Cost):
  Value: 0.5281

Interpretation:
  🟢🟢 Strong - Model discriminates very well

  (0.0 = no power, 1.0 = perfect discrimination)


## Cell 5: Severity Analysis (For Physicians with Claims)

In [5]:
print("\n" + "="*100)
print("SEVERITY ANALYSIS (Physicians With Claims)")
print("="*100)
print()

# Filter to physicians with claims
claims_df = test_df[test_df['kpi_total_frequency'] > 0].copy()

if len(claims_df) > 0:
    print(f"Physicians with claims: {len(claims_df):,} ({len(claims_df)/len(test_df)*100:.1f}% of test set)")
    print()
    
    # Average severity by composite score band
    severity_by_band = claims_df.groupby('composite_score').agg({
        'kpi_total_severity': ['count', 'mean', 'median', 'max'],
        'NPI': 'count'
    }).round(2)
    
    severity_by_band.columns = ['Claim Count', 'Avg Severity', 'Median Severity', 'Max Severity', '_']
    severity_by_band = severity_by_band.drop('_', axis=1)
    
    print("Severity by Composite Score Band (Claims Only):")
    print(severity_by_band)
    
    # Severity lift
    band_2_sev = claims_df[claims_df['composite_score'] == 2]['kpi_total_severity'].mean()
    band_high_sev = claims_df[claims_df['composite_score'] >= 7]['kpi_total_severity'].mean()
    
    if pd.notna(band_2_sev) and band_2_sev > 0:
        sev_lift = (band_high_sev / band_2_sev) if pd.notna(band_high_sev) else 0
        print(f"\nSeverity Lift:")
        print(f"  Low Risk (Band 2) Avg Severity:   {band_2_sev:,.2f}")
        print(f"  High Risk (Bands 7+) Avg Severity: {band_high_sev:,.2f}")
        print(f"  Severity Lift (High/Low):         {sev_lift:.2f}x")
else:
    print(f"⚠️  No physicians with claims in test set")


SEVERITY ANALYSIS (Physicians With Claims)

Physicians with claims: 1,790 (3.6% of test set)

Severity by Composite Score Band (Claims Only):
                 Claim Count  Avg Severity  Median Severity  Max Severity
composite_score                                                          
3                         15      39617.82          1771.67     497487.03
4                        239     128978.65          7199.40    2056284.83
5                        609     140380.78          5093.63    2255275.36
6                        743     220607.97         26000.78    2159293.17
7                        184     280113.65        114954.98    3521854.74


## Cell 6: Summary Report

In [6]:
print("\n" + "="*100)
print("VALIDATION SUMMARY")
print("="*100)
print()

# Initialize variables with defaults (in case cells ran out of order)
overall_lift = 0.0
decile_lift = 0.0

# Recalculate lift metrics if needed
try:
    band_1_loss = test_df[test_df['composite_score'] == 2]['kpi_total_loss_cost'].mean()
    band_high_loss = test_df[test_df['composite_score'] >= 7]['kpi_total_loss_cost'].mean()
    if pd.notna(band_1_loss) and band_1_loss > 0:
        overall_lift = (band_high_loss / band_1_loss) if pd.notna(band_high_loss) else 0
except:
    overall_lift = 0.0

try:
    test_df_decile = test_df.copy()
    test_df_decile['decile'] = pd.qcut(test_df_decile['composite_score'], q=10, labels=False, duplicates='drop')
    decile_means = test_df_decile.groupby('decile')['kpi_total_loss_cost'].mean()
    if len(decile_means) >= 2:
        top_decile_loss = decile_means.iloc[-1]
        bottom_decile_loss = decile_means.iloc[0]
        if pd.notna(bottom_decile_loss) and bottom_decile_loss > 0:
            decile_lift = top_decile_loss / bottom_decile_loss
except:
    decile_lift = 0.0

print("Data Split:")
print(f"  Train: {len(train_df):,} physicians")
print(f"  Test:  {len(test_df):,} physicians")

print(f"\nLoss Cost Metrics (Test Set):")
total_loss = test_df['kpi_total_loss_cost'].sum()
avg_loss = test_df['kpi_total_loss_cost'].mean()
print(f"  Total Loss Cost: ${total_loss:,.2f}")
print(f"  Average Loss Cost: ${avg_loss:,.2f}")
print(f"  Median Loss Cost: ${test_df['kpi_total_loss_cost'].median():,.2f}")

print(f"\nModel Performance:")
print(f"  ✅ Gini Coefficient: {gini_loss:.4f}")
if overall_lift > 0:
    print(f"  ✅ Band Lift: {overall_lift:.2f}x (Low risk to high risk)")
else:
    print(f"  ⚠️  Band Lift: Not calculated (check data)")
if decile_lift > 0:
    print(f"  ✅ Decile Lift: {decile_lift:.2f}x (Top to bottom decile)")
else:
    print(f"  ⚠️  Decile Lift: Not calculated (check data)")

print(f"\nConclusion:")
if decile_lift > 2.0 and gini_loss > 0.3:
    print(f"  🟢 Model shows STRONG discriminative power")
elif decile_lift > 1.5 and gini_loss > 0.2:
    print(f"  🟡 Model shows MODERATE discriminative power")
else:
    print(f"  🔴 Model shows WEAK discriminative power - review needed")


VALIDATION SUMMARY

Data Split:
  Train: 115,808 physicians
  Test:  49,815 physicians

Loss Cost Metrics (Test Set):
  Total Loss Cost: $283,579,593.33
  Average Loss Cost: $5,692.65
  Median Loss Cost: $0.00

Model Performance:
  ✅ Gini Coefficient: 0.5281
  ⚠️  Band Lift: Not calculated (check data)
  ✅ Decile Lift: 416.85x (Top to bottom decile)

Conclusion:
  🟢 Model shows STRONG discriminative power


## Cell 7: Export Validation Results

In [7]:
import os
from pathlib import Path

# ============================================================================
# CONFIGURE OUTPUT PARAMETERS
# ============================================================================

# Output directory path
output_directory = r'C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output'

# Output filename (without path)
output_filename = 'validation_results_70_30_split.csv'

# ============================================================================

# Create validation results file
validation_export = test_df[[
    'NPI',
    'OL_RISK_SPECIALTY_DESC',
    'ST',
    'composite_score',
    'adequacy_score',
    'capacity_score',
    'appetite_score',
    'environment_score',
    'kpi_total_loss_cost',
    'kpi_total_frequency',
    'kpi_total_severity'
]].copy()

output_path = os.path.join(output_directory, output_filename)

print(f"Output Configuration:")
print(f"  Directory: {output_directory}")
print(f"  Filename:  {output_filename}")
print(f"  Full path: {output_path}")
print()

# Save to CSV
try:
    # Create directory if it doesn't exist
    Path(output_directory).mkdir(parents=True, exist_ok=True)
    
    # Write CSV with UTF-8 encoding
    validation_export.to_csv(output_path, index=False, encoding='utf-8')
    
    print(f"✅ SUCCESS!")
    print(f"   Exported {len(validation_export):,} test set records")
    print(f"   File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
    print(f"\n   Saved to: {output_path}")
    
except PermissionError:
    print(f"❌ PERMISSION DENIED - Cannot write to: {output_directory}")
    print(f"\n   Saving to outputs folder instead...")
    alt_path = f'/mnt/user-data/outputs/{output_filename}'
    validation_export.to_csv(alt_path, index=False, encoding='utf-8')
    print(f"   ✅ Saved to: {alt_path}")
    
except Exception as e:
    print(f"❌ ERROR: {e}")
    print(f"\n   Saving to outputs folder instead...")
    alt_path = f'/mnt/user-data/outputs/{output_filename}'
    validation_export.to_csv(alt_path, index=False, encoding='utf-8')
    print(f"   ✅ Saved to: {alt_path}")

Output Configuration:
  Directory: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output
  Filename:  validation_results_70_30_split.csv
  Full path: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output\validation_results_70_30_split.csv

✅ SUCCESS!
   Exported 49,815 test set records
   File size: 5.67 MB

   Saved to: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output\validation_results_70_30_split.csv


## Summary

**Validation Metrics Tested:**

1. **Loss Cost by Band** — Average loss cost at each composite score level
2. **Decile Lift** — How much better top decile vs bottom decile
3. **Gini Coefficient** — Overall discrimination power (0-1 scale)
4. **Severity Lift** — For physicians with claims, does score predict severity?

**Interpretation:**
- Decile Lift > 2.0 = Strong model
- Gini > 0.3 = Moderate+ discrimination
- Severity Lift > 1.5 = Predicts severity reasonably